# Práctica: ANOVA, Chi-cuadrado y Pruebas No Paramétricas

En este cuaderno comparamos tres tratamientos médicos con ANOVA, estudiamos la asociación entre dos variables categóricas con un test Chi-cuadrado sobre una tabla de contingencia, y aplicamos Mann-Whitney U cuando los datos no siguen una distribución Normal.

In [ ]:
import sys
sys.path.append("../../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stats_toolkit import group_comparisons as gc

## 1. ANOVA: efectividad de 3 tratamientos médicos

Comparamos el tiempo de recuperación (en días) de pacientes bajo tres tratamientos distintos.

In [ ]:
rng = np.random.default_rng(5)
tratamiento_a = rng.normal(6, 1.2, size=15)
tratamiento_b = rng.normal(8.5, 1.2, size=15)
tratamiento_c = rng.normal(5, 1.2, size=15)

plt.boxplot([tratamiento_a, tratamiento_b, tratamiento_c], labels=["Tratamiento A", "Tratamiento B", "Tratamiento C"])
plt.ylabel("Días de recuperación")
plt.title("Tiempo de recuperación por tratamiento")
plt.show()

In [ ]:
f_stat, p_value = gc.anova_one_way(tratamiento_a, tratamiento_b, tratamiento_c)
print(f"F = {f_stat:.3f}, p-valor = {p_value:.6f}")

if p_value < 0.05:
    print("Al menos un tratamiento difiere significativamente de los demás.")
else:
    print("No hay evidencia suficiente de diferencias entre tratamientos.")

## 2. Chi-cuadrado: segmento de cliente vs. producto preferido

Construimos una tabla de contingencia a partir de datos simulados de clientes y comprobamos si el segmento de edad está asociado con el producto que prefieren.

In [ ]:
rng = np.random.default_rng(9)
n_clientes = 300

segmento = rng.choice(["Joven", "Adulto", "Senior"], size=n_clientes, p=[0.4, 0.35, 0.25])

# La preferencia depende del segmento (para que haya una asociación real que detectar)
prob_digital = {"Joven": 0.8, "Adulto": 0.5, "Senior": 0.2}
producto = [
    rng.choice(["Digital", "Físico"], p=[prob_digital[s], 1 - prob_digital[s]])
    for s in segmento
]

tabla = gc.build_contingency_table(segmento, producto)
tabla

In [ ]:
chi2_stat, p_value, dof, esperadas = gc.chi2_test_independence(tabla)
print(f"Chi² = {chi2_stat:.3f}, p-valor = {p_value:.6f}, g.l. = {dof}")

print("\nFrecuencias esperadas bajo independencia:")
print(pd.DataFrame(esperadas, index=tabla.index, columns=tabla.columns).round(1))

if p_value < 0.05:
    print("\nHay evidencia de asociación entre segmento y producto preferido.")
else:
    print("\nNo hay evidencia suficiente de asociación.")

## 3. Mann-Whitney U: valoraciones no normales de dos versiones de una app

Simulamos valoraciones (escala 1-10) claramente no normales: la versión 1 tiene valoraciones polarizadas (o encanta o decepciona), mientras que la versión 2 tiene valoraciones más consistentes.

In [ ]:
rng = np.random.default_rng(11)

# Distribución bimodal: mezcla de valoraciones muy bajas y muy altas
valoraciones_v1 = np.concatenate([
    rng.integers(1, 4, size=15),
    rng.integers(8, 11, size=15),
])
valoraciones_v2 = rng.integers(5, 8, size=30)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
axes[0].hist(valoraciones_v1, bins=range(1, 12), edgecolor="black")
axes[0].set_title("Versión 1 (polarizada)")
axes[1].hist(valoraciones_v2, bins=range(1, 12), edgecolor="black")
axes[1].set_title("Versión 2 (consistente)")
for ax in axes:
    ax.set_xlabel("Valoración")
plt.tight_layout()
plt.show()

In [ ]:
u_stat, p_value = gc.mann_whitney_u(valoraciones_v1, valoraciones_v2)
print(f"U = {u_stat:.3f}, p-valor = {p_value:.6f}")
print(f"Media v1: {valoraciones_v1.mean():.2f} | Media v2: {valoraciones_v2.mean():.2f}")

if p_value < 0.05:
    print("\nHay diferencia significativa entre las distribuciones de ambas versiones.")
else:
    print("\nNo hay evidencia suficiente de diferencia entre versiones.")

Fíjate en que ambas versiones pueden tener una media muy similar (por eso un t-test podría no detectar diferencia), pero Mann-Whitney sí puede detectar que sus **distribuciones** son distintas, porque compara la forma completa de los datos vía rangos, no solo la media.

## Ejercicios propuestos

1. Repite el ANOVA de la sección 1 pero con tres grupos generados con la MISMA media (por ejemplo, los tres con media 6). Comprueba que el p-valor ya no indica significancia.
2. Añade una cuarta categoría de producto ("Suscripción") a la tabla de contingencia de la sección 2 y repite el test Chi-cuadrado. ¿Cambian los grados de libertad? ¿Por qué?
3. Compara los resultados de aplicar un t-test estándar (`scipy.stats.ttest_ind`) frente a Mann-Whitney U sobre los datos polarizados de la sección 3. ¿Llegan a la misma conclusión? ¿Cuál te parece más apropiado dado que los datos no son normales?